# 03 稠相网络阈值分析

读取第二步生成的链对接触 NPZ，计算网络指标并绘制 PPT 图。居中与链对接触计算已移出 notebook。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, shortest_path
from IPython.display import Image, display

ROOT = Path("..")
if not (ROOT / "results").is_dir():
    raise FileNotFoundError("请从本文件夹的 notebook/ 目录启动 Jupyter")
RESULTS = ROOT / "results"
OUTPUT_DIR = ROOT / "figures/03_network"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VARIANTS = ("A18", "B01", "B10")
COLORS = {"A18": "#858a92", "B01": "#3774b7", "B10": "#ce7137"}
THRESHOLDS = (1, 5, 10, 20, 30, 35, 40, 50, 75, 100)
BLOCK_FRAMES = 50


### 网络指标计算


In [ ]:
def graph_stats(adjacency: np.ndarray) -> tuple[float, float]:
    """Return global efficiency and largest-component share."""
    n = len(adjacency)
    if n < 2:
        return np.nan, np.nan
    distances = shortest_path(csr_matrix(adjacency), directed=False,
                              unweighted=True)
    inv = np.zeros_like(distances, dtype=float)
    finite = np.isfinite(distances) & (distances > 0)
    inv[finite] = 1.0 / distances[finite]
    efficiency = float(inv.sum() / (n * (n - 1)))
    count, labels = connected_components(csr_matrix(adjacency), directed=False)
    largest_fraction = float(np.bincount(labels, minlength=count).max() / n)
    return efficiency, largest_fraction

def analyze_variant(variant: str, path: Path, block_frames: int) -> pd.DataFrame:
    with np.load(path) as saved:
        contacts = saved["contact_counts"]
        dense_pairs = saved["both_dense"]
    n_pairs = contacts.shape[1]
    n_chains = round((1 + np.sqrt(1 + 8 * n_pairs)) / 2)
    if n_chains * (n_chains - 1) // 2 != n_pairs:
        raise ValueError(f"{variant}: unexpected chain-pair count")
    pair_i, pair_j = np.triu_indices(n_chains, 1)
    rows = []
    for fi, (count, eligible) in enumerate(zip(contacts, dense_pairs)):
        dense = np.zeros(n_chains, dtype=bool)
        dense[pair_i[eligible]] = True
        dense[pair_j[eligible]] = True
        indices = np.flatnonzero(dense)
        for threshold in THRESHOLDS:
            pair_edge = eligible & (count >= threshold)
            full = np.zeros((n_chains, n_chains), dtype=bool)
            full[pair_i[pair_edge], pair_j[pair_edge]] = True
            full[pair_j[pair_edge], pair_i[pair_edge]] = True
            efficiency, giant = graph_stats(
                full[np.ix_(indices, indices)])
            rows.append({
                "variant": variant, "frame": fi, "block": fi // block_frames,
                "threshold": threshold,
                "global_efficiency": efficiency,
                "largest_component_fraction": giant,
            })
    return pd.DataFrame(rows)


### 从 NPZ 计算 PPT 图所需的时间块网络指标


In [ ]:
rows = pd.concat([
    analyze_variant(v, RESULTS / f"{v}_edge_arrays.npz", BLOCK_FRAMES)
    for v in VARIANTS
], ignore_index=True)
assert all(rows.loc[rows.variant.eq(v), "frame"].nunique() == 750 for v in VARIANTS)
blocks = rows.groupby(["variant", "threshold", "block"], as_index=False)[
    ["global_efficiency", "largest_component_fraction"]
].mean()


### PPT 用双面图


In [ ]:
"""Slide-size Python figure for the T260 graph-threshold result.

Core question: does the B01/B10 ordering depend on how many residue contacts
define a chain edge? Panel a is the primary result (efficiency), panel b checks
whether stronger filtering fragments the network. Block-SD is descriptive,
not replicate-based uncertainty. White, wide, editable-text SVG/PDF export.
"""

OUTPUT = OUTPUT_DIR / "graph_threshold_for_slide"
ppt_blocks = blocks[blocks.threshold <= 75]

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "svg.fonttype": "none", "pdf.fonttype": 42, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False,
})

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.35), constrained_layout=True)
for ax, metric, label in zip(
    axes,
    ("global_efficiency", "largest_component_fraction"),
    ("Global efficiency", "Largest component / dense chains"),
):
    for variant in ("A18", "B01", "B10"):
        sub = ppt_blocks[ppt_blocks.variant == variant]
        stat = sub.groupby("threshold")[metric].agg(["mean", "std"])
        x = stat.index.to_numpy(dtype=float)
        y = stat["mean"].to_numpy()
        sd = stat["std"].to_numpy()
        ax.plot(x, y, "-o", ms=4.2, lw=2.1, color=COLORS[variant],
                label=variant)
        ax.fill_between(x, np.maximum(y - sd, 0), np.minimum(y + sd, 1),
                        color=COLORS[variant], alpha=0.10, linewidth=0)
    ax.set(xlabel="Minimum residue contacts per chain pair", ylabel=label,
           xlim=(0, 77), xticks=[1, 10, 20, 30, 40, 50, 75])
    ax.grid(axis="y", alpha=0.2)
axes[0].set_ylim(0, 0.62)
axes[1].set_ylim(0, 1.04)
axes[0].legend(loc="upper right", title="Variant")
for label, ax in zip("ab", axes):
    ax.text(-0.12, 1.05, label, transform=ax.transAxes,
            fontweight="bold", fontsize=11)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT.with_suffix(".png"), dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(OUTPUT.with_suffix(".pdf"), bbox_inches="tight")
plt.close(fig)


### 查看图

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "graph_threshold_for_slide.png")))


### 单帧网络示意：接触阈值如何改变连通性

下面从已有 NPZ 读取第 375 个分析帧（`frame=374`），对每个变体固定同一组稠相链和同一套绘图坐标，比较链对至少有 5、30、50 个残基接触时的网络。节点代表稠相链，线代表达到阈值的链对；彩色节点属于最大连通分量，灰色节点属于其余分量。节点由固定的网络布局排布，并加入排斥力让它们更均匀。图上两点靠得近可能与接触关系有关，也可能只是布局造成；**位置与距离都不是模拟盒中的空间位置或物理密度**，不能据此判断链在空间中聚集。

每格标出边数、全局效率 `E` 与最大连通分量比例 `LCC`。这是一个单帧示例，用于解释上方两张统计图的含义；变体比较仍应以上方 750 帧的汇总为准。此单元格仅在 notebook 内显示图，不另存文件。

In [ ]:
from matplotlib.collections import LineCollection

SNAPSHOT_FRAME = 374
SNAPSHOT_THRESHOLDS = (5, 30, 50)

def load_dense_network(variant: str, frame: int) -> tuple[np.ndarray, np.ndarray]:
    """Return dense-chain contact counts for one frame."""
    with np.load(RESULTS / f"{variant}_edge_arrays.npz") as saved:
        counts = saved["contact_counts"][frame]
        eligible = saved["both_dense"][frame]
    n_pairs = len(counts)
    n_chains = round((1 + np.sqrt(1 + 8 * n_pairs)) / 2)
    if n_chains * (n_chains - 1) // 2 != n_pairs:
        raise ValueError(f"{variant}: unexpected chain-pair count")
    pair_i, pair_j = np.triu_indices(n_chains, 1)
    dense = np.zeros(n_chains, dtype=bool)
    dense[pair_i[eligible]] = True
    dense[pair_j[eligible]] = True
    if dense.sum() < 2:
        raise ValueError(f"{variant}, frame {frame}: fewer than two dense chains")
    full = np.zeros((n_chains, n_chains), dtype=np.int32)
    full[pair_i[eligible], pair_j[eligible]] = counts[eligible]
    full[pair_j[eligible], pair_i[eligible]] = counts[eligible]
    return full[np.ix_(dense, dense)], np.flatnonzero(dense)

def fixed_network_layout(contact_counts: np.ndarray) -> np.ndarray:
    """Make a fixed, more evenly spaced topology layout (not physical positions)."""
    adjacency = (contact_counts >= 1).astype(float)
    degree = adjacency.sum(axis=1)
    inv_sqrt = np.zeros_like(degree)
    inv_sqrt[degree > 0] = 1 / np.sqrt(degree[degree > 0])
    laplacian = np.eye(len(degree)) - (inv_sqrt[:, None] * adjacency * inv_sqrt[None, :])
    _, vectors = np.linalg.eigh(laplacian)
    xy = vectors[:, 1:3].copy()
    xy -= xy.mean(axis=0)
    scale = np.max(np.linalg.norm(xy, axis=1))
    if scale > 0:
        xy /= scale
    # Repel overlapping nodes while keeping connected nodes relatively close.
    target = np.sqrt(4 / len(xy))
    for iteration in range(180):
        delta = xy[:, None, :] - xy[None, :, :]
        distance = np.maximum(np.linalg.norm(delta, axis=-1), 1e-4)
        np.fill_diagonal(distance, 1.0)
        force = ((target**2 / distance - adjacency * distance**2 / target)
                 / distance)[..., None] * delta
        displacement = force.sum(axis=1)
        temperature = 0.08 * (1 - iteration / 180) + 0.002
        magnitude = np.maximum(np.linalg.norm(displacement, axis=1), 1e-12)
        xy += displacement / magnitude[:, None] * np.minimum(magnitude, temperature)[:, None]
        xy -= xy.mean(axis=0)
    xy /= np.max(np.linalg.norm(xy, axis=1))
    return xy

fig, axes = plt.subplots(len(VARIANTS), len(SNAPSHOT_THRESHOLDS),
                         figsize=(10.5, 9.0), constrained_layout=True)
for row, variant in enumerate(VARIANTS):
    contact_counts, _ = load_dense_network(variant, SNAPSHOT_FRAME)
    xy = fixed_network_layout(contact_counts)
    for col, threshold in enumerate(SNAPSHOT_THRESHOLDS):
        ax = axes[row, col]
        adjacency = contact_counts >= threshold
        edge_i, edge_j = np.where(np.triu(adjacency, 1))
        segments = np.stack((xy[edge_i], xy[edge_j]), axis=1)
        ax.add_collection(LineCollection(segments, colors=COLORS[variant],
                                         linewidths=0.65, alpha=0.18, zorder=1))
        _, labels = connected_components(csr_matrix(adjacency), directed=False)
        largest_label = np.bincount(labels).argmax()
        in_largest = labels == largest_label
        ax.scatter(xy[~in_largest, 0], xy[~in_largest, 1], s=24,
                   color="#bcbfc4", edgecolor="white", linewidth=0.35, zorder=2)
        ax.scatter(xy[in_largest, 0], xy[in_largest, 1], s=24,
                   color=COLORS[variant], edgecolor="white", linewidth=0.35, zorder=3)
        efficiency, giant = graph_stats(adjacency)
        ax.set(title=f"{variant} · threshold ≥ {threshold}\n"
                     f"{len(edge_i)} edges · E={efficiency:.2f} · LCC={giant:.2f}",
               xlim=(-1.08, 1.08), ylim=(-1.08, 1.08))
        ax.set_aspect("equal")
        ax.axis("off")
fig.suptitle(f"Example network at analysis frame {SNAPSHOT_FRAME + 1}", fontsize=13)
display(fig)
plt.close(fig)

### 可缩放的交互网络图

下方用 Plotly 显示同一分析帧。右上角菜单切换变体和接触阈值；滚轮缩放、拖动平移、悬停查看链编号与当前阈值下的邻居数。同一变体在不同阈值下使用相同节点位置。这里的节点坐标仍然只是绘图布局，不能作为物理距离使用。此单元格只在 notebook 中显示，不输出新文件。

In [ ]:
import plotly.graph_objects as go

states = []
for variant in VARIANTS:
    contact_counts, chain_ids = load_dense_network(variant, SNAPSHOT_FRAME)
    xy = fixed_network_layout(contact_counts)
    for threshold in SNAPSHOT_THRESHOLDS:
        adjacency = contact_counts >= threshold
        edge_i, edge_j = np.where(np.triu(adjacency, 1))
        _, labels = connected_components(csr_matrix(adjacency), directed=False)
        largest_label = np.bincount(labels).argmax()
        efficiency, giant = graph_stats(adjacency)
        edge_x, edge_y = [], []
        for i, j in zip(edge_i, edge_j):
            edge_x.extend((xy[i, 0], xy[j, 0], None))
            edge_y.extend((xy[i, 1], xy[j, 1], None))
        degree = adjacency.sum(axis=1)
        node_text = [
            f"Chain {chain_id} (0-based)<br>Partners at threshold {threshold}: {degree[i]}"
            f"<br>{'Largest component' if labels[i] == largest_label else 'Other component'}"
            for i, chain_id in enumerate(chain_ids)
        ]
        node_colors = [COLORS[variant] if label == largest_label else "#bcbfc4"
                       for label in labels]
        states.append({
            "label": f"{variant} · threshold ≥ {threshold}",
            "summary": f"{len(edge_i)} edges · E={efficiency:.2f} · LCC={giant:.2f}",
            "edge_x": edge_x, "edge_y": edge_y,
            "node_x": xy[:, 0], "node_y": xy[:, 1],
            "node_text": node_text, "node_colors": node_colors,
        })

interactive_fig = go.Figure()
for state_index, state in enumerate(states):
    shown = state_index == 0
    interactive_fig.add_trace(go.Scatter(
        x=state["edge_x"], y=state["edge_y"], mode="lines",
        line=dict(color="rgba(100, 105, 112, 0.25)", width=0.8),
        hoverinfo="skip", showlegend=False, visible=shown))
    interactive_fig.add_trace(go.Scatter(
        x=state["node_x"], y=state["node_y"], mode="markers",
        marker=dict(size=9, color=state["node_colors"],
                    line=dict(color="white", width=0.6)),
        text=state["node_text"], hovertemplate="%{text}<extra></extra>",
        showlegend=False, visible=shown))

buttons = []
for state_index, state in enumerate(states):
    visibility = [trace_index // 2 == state_index
                  for trace_index in range(2 * len(states))]
    buttons.append(dict(
        label=state["label"], method="update",
        args=[{"visible": visibility},
              {"title": f"{state['label']} · frame {SNAPSHOT_FRAME + 1}<br>"
                        f"<sup>{state['summary']}</sup>"}]))
interactive_fig.update_layout(
    title=f"{states[0]['label']} · frame {SNAPSHOT_FRAME + 1}<br>"
          f"<sup>{states[0]['summary']}</sup>",
    updatemenus=[dict(buttons=buttons, active=0, x=1, xanchor="right",
                      y=1.16, yanchor="top")],
    width=850, height=680, margin=dict(l=30, r=30, t=125, b=30),
    plot_bgcolor="white", hovermode="closest", uirevision="fixed-layout",
    xaxis=dict(visible=False, range=[-1.08, 1.08]),
    yaxis=dict(visible=False, range=[-1.08, 1.08], scaleanchor="x", scaleratio=1),
)
interactive_fig.show(config={"scrollZoom": True, "displaylogo": False})